# Testar o Clef-Flash no SageMaker

Este notebook baixa `Cloudflare/clef-flash` e executa uma decisão com probabilidades. Ele importa o arquivo `joint_schema_model.py` publicado junto ao modelo.

## Antes de executar

Escolha uma destas instâncias de notebook numa região compatível e configure pelo menos 80 GB de armazenamento. A AWS lista notebooks G5 na região de São Paulo; confirme a cota disponível na sua conta:

- **`ml.g6e.xlarge`**: uma GPU L40S com 48 GB de VRAM e 32 GiB de RAM.
- **`ml.g5.12xlarge`**: quatro GPUs A10G, cada uma com cerca de 22 GiB de VRAM, e 192 GiB de RAM. Este notebook distribui o modelo entre as GPUs disponíveis.

Execute as células em ordem usando um kernel Python com CUDA. O modelo foi testado pelos autores numa H200; estas opções têm VRAM suficiente, mas ainda precisam ser validadas com este código. A instância é cobrada enquanto estiver ligada. O armazenamento persistente pode continuar gerando cobrança depois que a instância for parada.

## 1. Conferir GPU, memória e disco

A célula interrompe a execução se o kernel não tiver CUDA ou espaço suficiente.

In [ ]:
import shutil
import torch
from pathlib import Path

print('PyTorch:', torch.__version__)
print('CUDA disponível:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Este kernel não tem PyTorch com CUDA. Selecione o ambiente GPU do SageMaker e reinicie o kernel.')

gpu_count = torch.cuda.device_count()
gpu_memory_gib = {i: torch.cuda.get_device_properties(i).total_memory / 1024**3 for i in range(gpu_count)}
for i, memory in gpu_memory_gib.items():
    print(f'GPU {i}: {torch.cuda.get_device_name(i)} ({memory:.1f} GiB VRAM)')
total_gpu_memory_gib = sum(gpu_memory_gib.values())
print(f'VRAM total: {total_gpu_memory_gib:.1f} GiB')
model_dir = Path.home() / 'SageMaker' / 'models' / 'clef-flash'
model_dir.parent.mkdir(parents=True, exist_ok=True)
disk_free_gb = shutil.disk_usage(model_dir.parent).free / 1024**3
print(f'Disco livre: {disk_free_gb:.1f} GiB')
if total_gpu_memory_gib < 30:
    raise RuntimeError('VRAM agregada insuficiente; use uma GPU de 30 GiB ou mais, ou várias GPUs com essa capacidade combinada.')
if disk_free_gb < 30:
    raise RuntimeError('Libere ou aumente o armazenamento antes de baixar o modelo.')

## 2. Instalar as dependências

O PyTorch do SageMaker não será substituído. Se o Jupyter pedir para reiniciar o kernel, reinicie e execute novamente a célula 1.

In [ ]:
%pip install "transformers==5.10.2" accelerate "huggingface_hub>=1.5,<2.0" pillow safetensors

## 3. Baixar o modelo

Os arquivos ficam em `~/SageMaker/models/clef-flash`, no armazenamento persistente do notebook.

In [ ]:
from huggingface_hub import snapshot_download

MODEL_ID = 'Cloudflare/clef-flash'
model_path = snapshot_download(
    repo_id=MODEL_ID,
    local_dir=model_dir,
    max_workers=4,
)
print('Modelo salvo em:', model_path)
print('Shards:', len(list(model_dir.glob('model-*-of-*.safetensors'))))

## 4. Carregar o modelo entre as GPUs

O loader publicado coloca tudo numa única GPU. Para instâncias com várias placas, usamos `device_map='auto'` do Transformers para distribuir as camadas do backbone; a cabeça Clef fica na GPU das embeddings de saída.

In [ ]:
import json
import sys
from safetensors.torch import load_file
from transformers import AutoProcessor, Qwen3_5ForConditionalGeneration

sys.path.insert(0, str(model_dir))
from joint_schema_model import JointSchemaHead, collate_records, encode_record

# Reserva aproximadamente 8 GiB por GPU para ativações e áreas temporárias.
max_memory = {i: f'{max(8, int(memory) - 8)}GiB' for i, memory in gpu_memory_gib.items()}
print('Limite de memória por GPU:', max_memory)

backbone = Qwen3_5ForConditionalGeneration.from_pretrained(
    model_dir,
    dtype=torch.bfloat16,
    device_map='auto',
    max_memory=max_memory,
    low_cpu_mem_usage=True,
)
backbone.config.use_cache = False
head_config = json.loads((model_dir / 'joint_head_config.json').read_text())
head = JointSchemaHead(**head_config)
head.load_state_dict(load_file(str(model_dir / 'joint_head.safetensors'), device='cpu'), strict=True)
output_weight = backbone.get_output_embeddings().weight
head = head.to(device=output_weight.device, dtype=torch.bfloat16).eval()

class ShardedClefModel(torch.nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head
        self.input_device = backbone.get_input_embeddings().weight.device
        self.output_device = backbone.get_output_embeddings().weight.device

    def forward(self, batch):
        base = self.backbone.get_base_model() if hasattr(self.backbone, 'get_base_model') else self.backbone
        media = batch.get('media') or {}
        text_model = base.model
        if not media and hasattr(text_model, 'language_model'):
            text_model = text_model.language_model
        input_ids = batch['input_ids'].to(self.input_device)
        attention_mask = batch['attention_mask'].to(self.input_device)
        media = {key: value.to(self.input_device) for key, value in media.items()}
        outputs = text_model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False, return_dict=True, **media)
        return self.head(
            outputs.last_hidden_state.to(self.output_device),
            input_ids.to(self.output_device),
            attention_mask.to(self.output_device),
            batch['records'],
            base.get_output_embeddings().weight,
        )

model = ShardedClefModel(backbone, head).eval()
print('Mapa das camadas:', backbone.hf_device_map)
print('GPU de entrada:', model.input_device, '| GPU da cabeça:', model.output_device)

## 5. Executar uma decisão

O exemplo pergunta se uma fatura está vencida e se o valor passa de US$ 1.000. A saída mostra probabilidades para cada opção.

In [ ]:
record = {
    'state': {'invoice': {'vendor': 'Acme', 'total': 1250.0, 'currency': 'USD', 'status': 'overdue'}},
    'questions': {
        'status': {
            'type': 'choice',
            'instructions': 'What is the invoice status?',
            'criteria': {'paid': 'Invoice is paid.', 'overdue': 'Invoice is past due.', 'draft': 'Not sent.'},
        },
        'large': {'type': 'noul', 'instructions': 'Is the total above 1000 USD?'},
    },
}

encoded = encode_record(processor.tokenizer, record, processor=processor)
batch = collate_records([encoded], processor.tokenizer.pad_token_id, model.input_device)
with torch.inference_mode():
    logits = model(batch)[0]

for question, question_logits in zip(encoded.questions, logits):
    probabilities = question_logits.float().softmax(-1).tolist()
    print(question.question_id, dict(zip(question.option_ids, probabilities)))

## 6. Parar a instância

Depois de guardar a saída, pare a instância pelo console do SageMaker para interromper a cobrança de computação. A pasta `~/SageMaker` e os pesos permanecem no armazenamento persistente; o armazenamento anexado ainda pode ser cobrado enquanto a instância estiver parada.